In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_13\messy_listings.csv")

In [3]:
df.shape

(50, 17)

In [4]:
df.dtypes

ListingID               str
HostName                str
PropertyType            str
Country                 str
City                    str
Address                 str
PricePerNight           str
Amenities               str
CreatedAt               str
CheckInDate             str
CheckOutDate            str
Nights              float64
HostResponseTime        str
Rating                  str
NumReviews              str
IsSuperhost             str
Notes                   str
dtype: object

In [5]:
df.head(5)

,ListingID,HostName,PropertyType,Country,City,Address,PricePerNight,Amenities,CreatedAt,CheckInDate,CheckOutDate,Nights,HostResponseTime,Rating,NumReviews,IsSuperhost,Notes
0,L001,Lucia Weber,House,Spain,Barcelona,238 River Rd,247.53,"['Balcony', 'Heating']",2024-01-01T07:15:00+01:00,2024-07-17,2024-07-27,10.0,within an hour,4.3,184,No,NaN
1,L002,Yusuf Weber,Apartment,USA,Munich,787 River Rd,236.99,"['Air Conditioning', 'Parking', 'Kitchen', 'He...",2024-03-04T07:45:00+00:00,2024-07-12,2024-07-18,6.0,within a day,4.7,187,1,NaN
2,L003,Ivy Chen,Villa,Portugal,Lisbon,310 King's Way,203.77,"['WiFi', 'Kitchen']",2024-03-03T19:15:00,2024-07-13,2024-07-18,5.0,a few days or more,4.5,94,Yes,NaN
3,L004,Liam Hollis,Condo,Canada,Vancouver,728 Oak Ave,$450.00,"['Gym', 'Kitchen', 'Parking']",2024-04-13T10:15:00-04:00,2024-09-11,2024-09-12,1.0,within a few hours,4.7,207,No,NaN
4,L005,Anna Bauer,Villa,United Kingdom,Edinburgh,907 King's Way,103.47,NaN,2024-02-24T14:30:00+00:00,2024-09-19,2024-09-26,7.0,a few days or more,4.0,36,No,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ListingID         50 non-null     str    
 1   HostName          50 non-null     str    
 2   PropertyType      50 non-null     str    
 3   Country           50 non-null     str    
 4   City              50 non-null     str    
 5   Address           50 non-null     str    
 6   PricePerNight     50 non-null     str    
 7   Amenities         49 non-null     str    
 8   CreatedAt         50 non-null     str    
 9   CheckInDate       50 non-null     str    
 10  CheckOutDate      50 non-null     str    
 11  Nights            49 non-null     float64
 12  HostResponseTime  49 non-null     str    
 13  Rating            49 non-null     str    
 14  NumReviews        49 non-null     str    
 15  IsSuperhost       50 non-null     str    
 16  Notes             2 non-null      str    
dtypes: float64

## PropertyType Normalization & Typo Correction

**Issue Identified:**  
The `PropertyType` column contains inconsistent capitalization (e.g., `"HOUSE"`, `"studio"`) and misspelling errors (e.g., `"aparment"`), creating redundant categories during aggregation.

**Fix Applied:**  
1. **Typo Correction:** Fixed misspellings (mapping `"aparment"` to `"Apartment"`).
2. **Title Casing Applied:** Applied `.str.title()` to standardize all property categories into consistent initial-capitalized strings.

In [7]:
df["PropertyType"] = df["PropertyType"].astype(str).str.strip().str.title()
df["PropertyType"]

0         House
1     Apartment
2         Villa
3         Condo
4         Villa
5      Aparment
6         Villa
7     Apartment
8         House
9         Villa
10    Apartment
11    Apartment
12        House
13        Condo
14       Studio
15        Condo
16        House
17        House
18       Studio
19        Villa
20        Condo
21        Villa
22    Apartment
23        Villa
24        Condo
25        House
26        Villa
27    Apartment
28    Apartment
29    Apartment
30    Apartment
31        Condo
32        Condo
33       Studio
34        Condo
35       Studio
36       Studio
37    Apartment
38        House
39        Villa
40       Studio
41        Villa
42        House
43       Studio
44        Villa
45    Apartment
46    Apartment
47        House
48        Condo
49    Apartment
Name: PropertyType, dtype: str

## Country Name Normalization & Whitelist Validation

**Objective:**  
Standardize country entries, resolve known typos/abbreviations, and coerce unverified or fictional country records (e.g., `"Atlantis"`) to `np.nan`.

**Methodology Applied:**  
1. **Case Normalization & Cleaning:** Converted all string values to lower case and trimmed leading/trailing whitespace using `.str.lower().str.strip()`.
2. **Explicit Typo & Alias Mapping:** Mapped known abbreviations and misspellings (`"usa"`, `"untied states"`, `"u.k."`, `"itali"`, `"nederland"`, `"mexicoo"`) to standard title-cased names via `country_map`.
3. **Fallback Title Casing:** Applied `.fillna(df["Country"].str.title())` so that valid, unmapped entries (e.g., `"Spain"`, `"Portugal"`) retain proper capitalization instead of becoming `NaN`.
4. **Whitelist Verification & Anomaly Coercion:** Checked entries against a list of approved countries (`valid_countries`). Any value not on the list was flagged and replaced with `np.nan`.

In [8]:
country_map = {
        'usa': 'United States',
        'untied states': 'United States',
        'u.k.': 'United Kingdom',
        'uk': 'United Kingdom',
        'germany': 'Germany',  
        'itali': 'Italy',
        'nederland': 'Netherlands',
        'mexicoo': 'Mexico',
}

df["Country"] = df["Country"].str.lower()
df["Country"] = df["Country"].str.strip()
df["Country"]= df["Country"].map(country_map).fillna(df["Country"].str.title())
df["Country"]

0              Spain
1      United States
2           Portugal
3             Canada
4     United Kingdom
5              Spain
6              Italy
7      United States
8             Mexico
9             Mexico
10     United States
11    United Kingdom
12     United States
13           Germany
14    United Kingdom
15            Canada
16          Portugal
17       Netherlands
18           Germany
19            Canada
20             Italy
21             Italy
22           Germany
23    United Kingdom
24            France
25            France
26             Italy
27    United Kingdom
28          Portugal
29             Italy
30             Spain
31            Mexico
32             Italy
33           Germany
34           Germany
35             Spain
36       Netherlands
37            France
38    United Kingdom
39          Portugal
40            France
41            Canada
42             Spain
43    United Kingdom
44            Mexico
45          Portugal
46     United States
47           

In [9]:
valid_countries = ['United States','Spain','France','Germany','Italy',
                    'Canada','United Kingdom','Mexico','Portugal','Netherlands']

invalid_country_mask = ~df["Country"].isin(valid_countries)
df.loc[invalid_country_mask, ['ListingID','HostName','Country','City']]

,ListingID,HostName,Country,City
48,L049,Sarah Voss,Atlantis,Milan


In [10]:
df.loc[invalid_country_mask, "Country"] = np.nan
df["Country"].isna().sum()

np.int64(1)

## Identifying Potential Duplicate Listings

**Objective:**  
Detect near-duplicate records that likely represent the same listing/host but differ slightly due to minor typos, variations in naming (e.g., `"Sarah Connor"` vs. `"Sara Conner"`), or subtle address differences (e.g., `"482 Oak Ave"` vs. `"482 Oak Ave."`).

**Methodology Applied:**  
1. **String Similarity Metric:** Utilized `difflib.SequenceMatcher` to compute normalized similarity scores ($[0.0, 1.0]$) across lowercased `HostName` and `Address` fields.
2. **Pairwise Comparison:** Performed exhaustive upper-triangle pairwise matching ($(i, j)$ where $j > i$) across the dataset.
3. **Multi-Threshold Matching:**
   * Host Name similarity threshold: $\text{ratio} > 0.75$
   * Address similarity threshold: $\text{ratio} > 0.85$
4. **Candidate Inspection:** Extracted identified duplicate pairs alongside timestamps (`CreatedAt`) to determine primary records vs. redundant entries.

In [11]:
import ast
from difflib import SequenceMatcher

def similar(a,b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

candidates = []
for i in range(len(df)):
    for j in range(i+1, len(df)):
        name_sim = similar(df.loc[i, 'HostName'], df.loc[j, 'HostName'])
        addrs_sim = similar(df.loc[i, 'Address'], df.loc[j, 'Address'])
        if name_sim > 0.75 and addrs_sim > 0.85:
            candidates.append((i,j, round(name_sim, 2), round(addrs_sim, 2)))

candidates

[(10, 46, 0.87, 0.96)]

In [14]:
i, j, _, _ = candidates[0]
df.loc[[i,j], ['ListingID', 'HostName', 'Address', 'CreatedAt']]

,ListingID,HostName,Address,CreatedAt
10,L011,Sarah Connor,482 Oak Ave,2024-03-10T09:00:00
46,L047,Sara Conner,482 Oak Ave.,2024-05-02T09:00:00Z


In [16]:
drop_indices = [j for _, j, _, _ in candidates]

drop_indices = list(set(drop_indices))

print(f"Dropping {len(drop_indices)} duplicate row(s): {drop_indices}")
display(df.loc[drop_indices, ['ListingID', 'HostName', 'Address', 'CreatedAt']])

df = df.drop(index=drop_indices).reset_index(drop=True)

print(f"Remaining records in dataset: {len(df)}")

Dropping 1 duplicate row(s): [46]


,ListingID,HostName,Address,CreatedAt
46,L048,Leo Costa,260 Via Roma,2024-05-28T12:15:00+01:00


Remaining records in dataset: 48
